# Team Selection UI
This notebook sets up the interactive team selection interface.

In [1]:
import json
import ipywidgets as widgets
from IPython.display import display

# Load eligible players data
with open('eligible_players.json', 'r') as f:
    eligible = json.load(f)['players']

# Prepare selected_players template
selected_players = {
    'C': {'id': '', 'name': '', 'lineup_slot': ''},
    '1B': {'id': '', 'name': '', 'lineup_slot': ''},
    '2B': {'id': '', 'name': '', 'lineup_slot': ''},
    '3B': {'id': '', 'name': '', 'lineup_slot': ''},
    'SS': {'id': '', 'name': '', 'lineup_slot': ''},
    'OF': [{'id': '', 'name': '', 'lineup_slot': ''} for _ in range(3)],
    'DH': {'id': '', 'name': '', 'lineup_slot': ''},
    'PH': [{'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''} for _ in range(6)]
}

# Build player name lists by position
player_names_by_position = {}
for pos in ['C', '1B', '2B', '3B', 'SS', 'OF', 'DH']:
    player_names_by_position[pos] = ['Select Player'] + [p['longName'] for p in eligible if p['pos'] == pos]
# PH can be any non-pitcher
player_names_by_position['PH'] = ['Select Player'] + [p['longName'] for p in eligible if p['pos'] != 'P']

# Define dropdowns for starters
positions = ['C', '1B', '2B', '3B', 'SS', 'OF', 'OF', 'OF', 'DH']
dropdowns = [widgets.Dropdown(options=player_names_by_position[pos], description=pos) for pos in positions]

# Dropdowns for pinch hitters
bench_dropdowns = [widgets.Dropdown(options=player_names_by_position['PH'], description=f'PH{i+1}') for i in range(6)]

# Button and output
button = widgets.Button(description='Select team')
output = widgets.Output()

def on_select(b):
    output.clear_output()
    names = [d.value for d in dropdowns] + [b.value for b in bench_dropdowns]
    # Validation
    if 'Select Player' in names:
        with output:
            print('Error: All slots must be filled.')
        return
    if len(names) != len(set(names)):
        with output:
            print('Error: Duplicate player selected.')
        return
    # Assign selections
    # Starters
    idx_map = {pos: [] for pos in positions}
    for dd in dropdowns:
        idx_map[dd.description].append(dd.value)
    for pos in ['C','1B','2B','3B','SS','DH']:
        name = idx_map[pos][0]
        pid = next(p['playerID'] for p in eligible if p['longName']==name)
        selected_players[pos].update({'id': pid, 'name': name})
    # OFs
    for i, name in enumerate(idx_map['OF']):
        pid = next(p['playerID'] for p in eligible if p['longName']==name)
        selected_players['OF'][i].update({'id': pid, 'name': name})
    # Pinch hitters
    for i, dd in enumerate(bench_dropdowns):
        name = dd.value
        pid = next(p['playerID'] for p in eligible if p['longName']==name)
        selected_players['PH'][i].update({'id': pid, 'name': name, 'position': 'PH', 'bench_order': i+1})
    with output:
        print(selected_players)

button.on_click(on_select)

# Display UI
display(*dropdowns, widgets.HTML('<br>'), *bench_dropdowns, button, output)

Dropdown(description='C', options=('Select Player', 'Jacob Stallings', 'Henry Davis', 'Jonah Heim', 'Austin Ba…

Dropdown(description='1B', options=('Select Player', 'Kody Clemens', 'Justin Turner', 'Paul Goldschmidt', 'Jon…

Dropdown(description='2B', options=('Select Player', 'Nicky Lopez', 'Gavin Lux', 'Amed Rosario', 'Jonathan Ind…

Dropdown(description='3B', options=('Select Player', 'Chase Meidroth', 'Gage Workman', 'Miguel Vargas', 'Ramón…

Dropdown(description='SS', options=('Select Player', 'Nick Allen', 'Edmundo Sosa', 'Kevin Newman', 'Jacob Amay…

...

Dropdown(description='PH4', options=('Select Player', 'Blake Dunn', 'Nick Allen', 'Bryan De La Cruz', 'Chase M…

Dropdown(description='PH5', options=('Select Player', 'Blake Dunn', 'Nick Allen', 'Bryan De La Cruz', 'Chase M…

Dropdown(description='PH6', options=('Select Player', 'Blake Dunn', 'Nick Allen', 'Bryan De La Cruz', 'Chase M…

Button(description='Select team', style=ButtonStyle())

Output()